# PKU training matrix on GPU

Runs the whole Step 4 matrix for PKU-Market-PCB in one go: the nine student runs
(input modes A/B/C x three seeds), the three ResNet-18 teachers, and the six KD runs
that distil from them.

| | on this project's CPU box | why it matters |
| --- | --- | --- |
| student epoch | ~93 s | the 9-run matrix is ~7 hours |
| teacher epoch | ~1.6 h | ~2 days per seed, so the KD rows cannot be run at all |

The teachers are the point. Without them there is no Hinton-KD comparison and no
defect-aware KD -- which is the project's novel contribution -- so the KD half of the
matrix simply does not exist off-GPU.

**Before running:** Runtime -> Change runtime type -> **GPU**.


## 1. Setup


In [ ]:
import os
import subprocess
import sys

import torch

BRANCH = "pku-dataset"
REPO = "https://github.com/vsubacius/PCBerrorDetection.git"

if not os.path.isdir("PCBerrorDetection"):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO], check=True)
os.chdir(os.path.abspath("PCBerrorDetection"))

print("cwd  ", os.getcwd())
print("torch", torch.__version__)
print("GPU  ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
if not torch.cuda.is_available():
    raise SystemExit("Set Runtime -> Change runtime type -> GPU before running this.")


## 2. Results folder on Drive

Colab runtimes recycle, and a disconnect loses everything in the container. `runs_pku/`
is symlinked to Drive so every finished run survives, and re-running this notebook
resumes rather than restarts -- `run_matrix.py` skips any run that already has a
`metrics.json`, and `train.py` resumes a half-finished one from its last epoch.


In [ ]:
import pathlib

from google.colab import drive

drive.mount("/content/drive")
DRIVE_RUNS = pathlib.Path("/content/drive/MyDrive/pcb/runs_pku")
DRIVE_RUNS.mkdir(parents=True, exist_ok=True)

local = pathlib.Path("runs_pku")
if not local.is_symlink() and not local.exists():
    local.symlink_to(DRIVE_RUNS, target_is_directory=True)
print("runs_pku ->", os.path.realpath("runs_pku"))
print("already finished:", sorted(p.parent.name for p in DRIVE_RUNS.glob("*/metrics.json")))


## 3. The tile dataset

Either copy the built `.npz` (79 MB) from Drive, or rebuild it here from the PKU
mirror. Rebuilding clones 3.7 GB and re-runs the converter, about 20-30 minutes; it is
deterministic, so both routes give the same crops and the same board-level folds.


In [ ]:
NPZ = pathlib.Path("data/tiles_pku/pku_tiles_t160_s120.npz")
DRIVE_NPZ = pathlib.Path("/content/drive/MyDrive/pcb/pku_tiles_t160_s120.npz")

if not NPZ.exists() and DRIVE_NPZ.exists():
    NPZ.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["cp", str(DRIVE_NPZ), str(NPZ)], check=True)
    print("copied from Drive")

if not NPZ.exists():
    subprocess.run([sys.executable, "data/pku.py", "--download"], check=True)
    subprocess.run([sys.executable, "data/pku.py", "--convert"], check=True)
    subprocess.run([sys.executable, "data/build_tiles.py", "--dataset", "pku"], check=True)
    DRIVE_NPZ.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["cp", str(NPZ), str(DRIVE_NPZ)], check=True)
    print("built and cached to Drive")

print("tile dataset:", NPZ, f"{NPZ.stat().st_size / 1e6:.0f} MB")


## 4. Run the matrix

Order matters: the KD rows need the teachers, and `run_matrix.py` refuses to start
them if a teacher checkpoint is missing rather than silently distilling from nothing.

`--out-dir runs_pku` keeps these apart from the DeepPCB results in `runs/`. The two
datasets have their own teachers; crossing them would be a silent, invisible error.


In [ ]:
def run_set(name):
    print(f"
{'=' * 60}
{name}
{'=' * 60}", flush=True)
    subprocess.run([sys.executable, "training/run_matrix.py",
                    "--set", name,
                    "--out-dir", "runs_pku",
                    "--npz", str(NPZ),
                    "--device", "cuda",
                    "--num-workers", "2"], check=True)

run_set("input_ablation")   # 9 student runs: modes A, B, C x seeds 0, 1, 2
run_set("teacher")          # 3 ResNet-18 teachers, the reason for the GPU
run_set("kd")               # 6 runs: Hinton KD and defect-aware KD per seed


## 5. Read the results against the right floor

PKU's validation fold is **0.60% positive per class**; DeepPCB's is 6.49%. Macro AP of
a random ranker equals the prevalence, so a PKU AP and a DeepPCB AP are not the same
quantity. Compare the **lift** across datasets, the raw AP only within one.


In [ ]:
import json

sys.path.insert(0, "eval")
sys.path.insert(0, "data")
import numpy as np
from metrics import average_precision_lift, tile_prevalence
from tile_dataset import TileData

val = TileData.load(str(NPZ), "val")
targets = np.concatenate([val.y[i] for i in range(len(val))])
ignore = np.concatenate([val.ignore[i] for i in range(len(val))]).astype(bool)
_, prevalence = tile_prevalence(targets, ignore)
print(f"PKU val prevalence {prevalence:.4%} (the macro AP of a random ranker)
")

print(f"{'run':<20}{'val mAP':>10}{'lift':>9}{'epochs':>8}")
for path in sorted(pathlib.Path("runs_pku").glob("*/metrics.json")):
    m = json.loads(path.read_text())
    ap = m.get("best_val_macro_ap")
    if ap is None:
        continue
    lift = average_precision_lift(ap, prevalence)
    print(f"{m['run_id']:<20}{ap:>10.4f}{lift:>8.1f}x{m['epochs_run']:>8}")


## 6. Bring the results back

Everything under `runs_pku/` is already on Drive via the symlink. Student checkpoints
are ~274 KB and are committed; teacher weights are ~45 MB and are **not**
(`.gitignore` excludes them). Download `runs_pku/` to the PC, then commit the configs,
metrics and student weights from there.
